# Evaluate TTA on multiple deepfake datasets

Notebook này chạy `code/testing/evaluate_tta_matrix.py` cho nhiều tập feature `.pt` hoặc folder feature. Sửa cell config bên dưới cho đúng path local/Kaggle rồi chạy từ trên xuống.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

import pandas as pd
import torch

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path('/kaggle/working/training-free-tta-for-deepfake-detection')]
    for root in candidates:
        if (root / 'code' / 'deepfake_tta').exists() and (root / 'code' / 'testing' / 'evaluate_tta_matrix.py').exists():
            return root.resolve()
    raise FileNotFoundError('Cannot find repo root containing code/deepfake_tta and code/testing/evaluate_tta_matrix.py')

REPO_ROOT = find_repo_root()
CODE_ROOT = REPO_ROOT / 'code'
SCRIPT = CODE_ROOT / 'testing' / 'evaluate_tta_matrix.py'
sys.path.insert(0, str(CODE_ROOT))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo:', REPO_ROOT)
print('device:', DEVICE)

## Config

- `TRAIN_FEATURES`: FF++ train CLIP embedding dùng để fit cache/prototype cho TTA.
- `DATASET_SPECS`: mapping `dataset_name -> file_or_folder`. Folder sẽ được script tự quét `*features.pt`.
- `MODEL_SPECS`: mapping `model_name -> model.pt`.
- Datasets trong `BALANCED_DATASETS` sẽ undersample từng file về số REAL/FAKE bằng nhau.
- Datasets trong `BALANCED_ALIGNED_DATASETS` sẽ lấy giao sample ID chung giữa các corruption level rồi cân bằng/aligned, hợp với so sánh corruptions.

In [ ]:
ON_KAGGLE = Path('/kaggle/input').exists()

if ON_KAGGLE:
    # Đổi các path này theo tên Kaggle dataset đang attach.
    FFPP_SPLIT = Path('/kaggle/input/ffpp-split-features')
    FFPP_CORR = Path('/kaggle/input/ffpp-test-embeddings/ffpp_test_embeddings')
    CELEB_CORR = Path('/kaggle/input/deepfakebench-features')
    MODELS_DIR = Path('/kaggle/input/ffpp-training-free-models')
    MODELS_BAL = Path('/kaggle/input/ffpp-training-free-models-balanced')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    # Local defaults. Sửa sang nơi bạn đặt feature nếu khác.
    FFPP_SPLIT = REPO_ROOT / 'embeddings' / 'ffpp-split-features'
    FFPP_CORR = REPO_ROOT / 'embeddings' / 'ffpp_test_embeddings'
    CELEB_CORR = REPO_ROOT / 'embeddings' / 'deepfakebench-features'
    MODELS_DIR = REPO_ROOT / 'models'
    MODELS_BAL = REPO_ROOT / 'models'
    OUTPUT_DIR = REPO_ROOT / 'eda'

TRAIN_FEATURES = FFPP_SPLIT / 'ffpp_train_features.pt'

DATASET_SPECS = {
    'ffpp-test': FFPP_SPLIT / 'ffpp_test_features.pt',
    'ffpp-test-corruption': FFPP_CORR,
    'celebdfv1-test-corruption': CELEB_CORR,
    'ffpp-test-balanced': FFPP_SPLIT / 'ffpp_test_features.pt',
    'ffpp-test-corruption-balanced': FFPP_CORR,
    'celebdfv1-test-corruption-balanced': CELEB_CORR,
}

MODEL_SPECS = {
    'linear-probe': MODELS_DIR / 'ffpp_linear_probe_split.pt',
    'osd': MODELS_DIR / 'ffpp_osd_linear_probe_split.pt',
    # Bật nếu có model balanced riêng.
    # 'linear-probe-balanced': MODELS_BAL / 'ffpp_linear_probe_split.pt',
    # 'osd-balanced': MODELS_BAL / 'ffpp_osd_linear_probe_split.pt',
}

TTA_METHODS = [
    'none',
    'dota',
    'bca',
    'adaptive_dota_bca',
    'freetta',
    'tip_adapter',
]

BALANCED_DATASETS = {'ffpp-test-balanced'}
BALANCED_ALIGNED_DATASETS = {'ffpp-test-corruption-balanced', 'celebdfv1-test-corruption-balanced'}

RESULTS_OUTPUT = OUTPUT_DIR / 'tta_all_datasets_matrix_results.csv'
SUMMARY_OUTPUT = OUTPUT_DIR / 'tta_all_datasets_matrix_summary.csv'

EVAL_BATCH_SIZE = 4096
TEST_BATCH_SIZE = 512
CACHE_BATCH_SIZE = 8192
BLOCK_SIZE = 16
SEED = 42
BALANCE_METHOD_FIT = True
CONTINUE_ON_ERROR = True
SHOW_REPORT = False

print('train:', TRAIN_FEATURES)
print('results:', RESULTS_OUTPUT)

In [ ]:
def exists_or_has_features(path):
    path = Path(path)
    return path.is_file() or (path.is_dir() and any(path.glob('**/*features.pt')))

missing = []
if not Path(TRAIN_FEATURES).is_file():
    missing.append(('train_features', TRAIN_FEATURES))

available_datasets = {}
for name, path in DATASET_SPECS.items():
    if exists_or_has_features(path):
        available_datasets[name] = Path(path)
    else:
        missing.append((f'dataset:{name}', path))

available_models = {}
for name, path in MODEL_SPECS.items():
    if Path(path).is_file():
        available_models[name] = Path(path)
    else:
        missing.append((f'model:{name}', path))

print('available datasets:')
for name, path in available_datasets.items():
    print(' -', name, '=', path)
print('\navailable models:')
for name, path in available_models.items():
    print(' -', name, '=', path)

if missing:
    print('\nmissing/skipped paths:')
    for name, path in missing:
        print(' -', name, '=', path)

assert Path(TRAIN_FEATURES).is_file(), 'Set TRAIN_FEATURES to an existing .pt file before running evaluation.'
assert available_datasets, 'No dataset feature file/folder found. Update DATASET_SPECS.'
assert available_models, 'No model .pt found. Update MODEL_SPECS.'

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    '--train-features', str(TRAIN_FEATURES),
    '--results-output', str(RESULTS_OUTPUT),
    '--device', DEVICE,
    '--seed', str(SEED),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--test-batch-size', str(TEST_BATCH_SIZE),
    '--cache-batch-size', str(CACHE_BATCH_SIZE),
    '--block-size', str(BLOCK_SIZE),
    '--tta-methods', *TTA_METHODS,
]

for name, path in available_datasets.items():
    cmd += ['--dataset', f'{name}={path}']
for name, path in available_models.items():
    cmd += ['--model', f'{name}={path}']
for name in sorted(BALANCED_DATASETS & set(available_datasets)):
    cmd += ['--balanced-dataset', name]
for name in sorted(BALANCED_ALIGNED_DATASETS & set(available_datasets)):
    cmd += ['--balanced-aligned-dataset', name]
if BALANCE_METHOD_FIT:
    cmd += ['--balance-method-fit']
if CONTINUE_ON_ERROR:
    cmd += ['--continue-on-error']
if SHOW_REPORT:
    cmd += ['--show-report']

print(' '.join(shlex.quote(part) for part in cmd))

In [ ]:
RESULTS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
completed = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True)
completed.check_returncode()

In [ ]:
results = pd.read_csv(RESULTS_OUTPUT)
display(results.head())
display(results.sort_values(['dataset', 'level', 'corruption', 'model', 'method'], na_position='first'))

In [ ]:
metric_cols = ['acc', 'f1', 'auc', 'ap', 'eer']
valid = results.copy()
if 'error' in valid.columns:
    valid = valid[valid['error'].isna()]

summary = (
    valid.groupby(['dataset', 'model', 'method'], dropna=False)[metric_cols]
    .mean()
    .reset_index()
    .sort_values(['dataset', 'model', 'f1'], ascending=[True, True, False])
)
summary.to_csv(SUMMARY_OUTPUT, index=False)
display(summary)
print('saved results:', RESULTS_OUTPUT)
print('saved summary:', SUMMARY_OUTPUT)

In [ ]:
best = (
    summary.sort_values(['dataset', 'model', 'f1'], ascending=[True, True, False])
    .groupby(['dataset', 'model'], as_index=False)
    .head(3)
)
display(best)